<a href="https://colab.research.google.com/github/gyeongee/TIL/blob/main/matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 행렬의 곱

행렬 A와 행렬 B를 곱할 때, A의 열의 개수와 B의 행의 개수는 같아야 합니다.

행렬의 곱셈 과정은 내적 또는 닷 프로덕트라고 부릅니다.



### 벡터 행렬 곱
벡터와 행렬의 곱셈도 행렬의 곱셈처럼 생각해볼 수 있습니다. 주의할 점은 벡터가 곱셈의 앞에 위치할 경우, 전치(transpose)를 통해 행과 열을 바꿔 표현하여 곱셈을 수행합니다.

### 행렬 곱

In [1]:
import torch
x=torch.FloatTensor([[1,2],
                     [3,4],
                     [5,6]])
y=torch.FloatTensor([[1,2],
                     [3,4]])
print(x.shape, y.shape)

z=torch.matmul(x,y)
print(z.size())

torch.Size([3, 2]) torch.Size([2, 2])
torch.Size([3, 2])


### 배치 행렬의 곱 : bmm()

딥러닝을 수행할 때 보통 여러 샘플을 동시에 병렬 계산하곤 합니다. 따라서 행렬 곱 연산의 경우에도 여러 곱셈을 동시에 진행할 수 있어야 합니다.

In [2]:
x=torch.FloatTensor(3,3,2)
y=torch.FloatTensor(3,2,3)

z=torch.bmm(x,y)
print(z.shape)

torch.Size([3, 3, 3])


3 x 3 크기의 행렬이 3개 있는 3 x 3 x 3 형태로 나오는 것을 볼 수 있습니다.

bmm()는 마지막 2개의 차원을 행렬 취급하여 병렬로 행렬 곱 연산을 수행합니다.


### 선형 계층

입력 벡터 x는 n차원의 실수 벡터이며, 출력 벡터 y는 m차원의 실수 벡터입니다.

따라서 n차원을 m차원으로 변환해주기 위해서 $W$는 n X m 차원의 행렬이 되어야 합니다.

다수의 입력을 처리하기 위해 병렬(parallel)연산을 진행합니다. 이를 미니배치(mini-batch)라 부릅니다.

병렬 연산을 통해 연산 속도를 높일 수 있으며, 이것은 GPU 상에서 행렬 곱셈으로 처리됩니다.

선형 계층은 행렬 곱셈과 벡터의 덧셈으로 이루어져 있기 때문에 선형 변환이라고 볼 수 있습니다.

### 선형 계층

선형 계층의 파라미터 행렬 $W$가 행렬 곱 연산에 활용될 것이고, 파라미터 벡터 b가 브로드캐스팅 덧셈 연산에 활용될 것입니다.

In [3]:
w=torch.FloatTensor([[1,2],
                     [3,4],
                     [5,6]])
# 원소가 두 개인 1차원 텐서
b=torch.FloatTensor([2,2])

print(w.shape,b.shape)

torch.Size([3, 2]) torch.Size([2])


In [4]:
def linear(x,w,b):
  y=torch.matmul(x,w)+b
  return y

In [5]:
x=torch.FloatTensor(4,3)

In [6]:
y=linear(x,w,b)
print(y.size())

torch.Size([4, 2])


### 계층 만들기

파이토치에는 nn 패키지가 있고 내부에는 미리 정의된 많은 신경망들이 있습니다.

그리고 그 신경망들은 `torch.nn.Module`이라는 추상 클래스를 상속받아 선형 계층을 구현할 수 있습니다.

`torch.nn.Parameter`클래스를 활용해 학습이 가능하도록 인식할 수 있는 파라미터로 만들어주겠습니다.

In [8]:
import torch.nn as nn


`nn.Module`을 상속받은 클래스는 보통 2개의 메서드,__init__과 forward를 오버라이드(override)합니다.

`__init__()`는 계층 내부에서 필요한 변수를 미리 선언하고 있으며 심지어 또 다른 계층(nn.Module을 상속받은 클래스의 객체)을 소유하도록 할 수도 있습니다.

forward()는 계층을 통과하는데 필요한 계산 수행을 합니다.


In [9]:
class MyLinear(nn.Module):
  def __init__(self,input_dim=3,output_dim=2):
    self.input_dim=input_dim
    self.output_dim=output_dim

    super().__init__()

    self.W=nn.Parameter(torch.FloatTensor(input_dim,output_dim))
    self.b=nn.Parameter(torch.FloatTensor(output_dim))

  #  You should override 'forward' method to implement detail.
  def forward(self,x):
    #|x|=(batch_size,input_dim)
    y=torch.matmul(x,self.W)+self.b
    #|y|=(batch_size,input_size)*(input_dim,output_dim)
    #   =(batch_size.output_size)
    return y

In [10]:
linear=MyLinear(3,2)
y=linear(x)

여기서 중요한 점은 forward()를 따로 호출하지 않고 객체명에 바로 괄호를 열어 텐서 x를 인수로 넘겨주었다는 것입니다.

이처럼 `nn.Module`의 상속을 받은 객체는 `call__()`와`forward`가 매핑되어 있어서 forward를 직접 부를 필요가 없습니다.

In [11]:
# 파리미터 출력
for p in linear.parameters():
  print(p)

Parameter containing:
tensor([[5.0963e-27, 0.0000e+00],
        [0.0000e+00, 0.0000e+00],
        [4.4842e-44, 0.0000e+00]], requires_grad=True)
Parameter containing:
tensor([0., 0.], requires_grad=True)


### nn.Linear() 활용하기

`torch.nn`에 미리 정의된 선형 계층을 사용합시다.

In [12]:
linear=nn.Linear(3,2)
y=linear(x)

for p in linear.parameters():
  print(p)

Parameter containing:
tensor([[ 0.3885,  0.5456,  0.5152],
        [-0.3061,  0.2353, -0.3540]], requires_grad=True)
Parameter containing:
tensor([-0.3698, -0.3813], requires_grad=True)


In [13]:
class MyLinear(nn.Module):
  def __init__(self,input_dim=3,output_dim=2):
    self.input_dim=input_dim
    self.output_dim=output_dim

    super().__init__()

    self.linear=nn.Linear(input_dim.output_dim)

  def forward(self,x):
    y=self.linear(x)

    return y

### GPU 사용하기

In [14]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
# # 맥북 내장 gpu 사용하기
# device="cuda" if torch.backends.mps.is_available() else "cpu"
# print(device)

In [15]:
x=torch.cuda.FloatTensor(2,2)
x

/tmp/ipykernel_2791/3064550110.py:1: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  x=torch.cuda.FloatTensor(2,2)


tensor([[0., 0.],
        [0., 0.]], device='cuda:0')

In [16]:
#cuda()를 통해 CPU 메모리 상에 선언된 텐서 GPU로 복사하는 방법 존재
x=torch.FloatTensor(2,2)
x=x.cuda()
x

tensor([[1.4061e-40, 0.0000e+00],
        [0.0000e+00, 0.0000e+00]], device='cuda:0')

In [17]:
#  cuda 함수의 인자에 복사하고자 하는 목적지 GPU 장치의 인덱스를 넣어 원하는 디바이스에 복사할 수 있습니다
x=x.cuda(device=1)
x

AcceleratorError: CUDA error: invalid device ordinal
GPU device may be out of range, do you have enough GPUs?
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


Colab은 보통 GPU가 1개(인덱스 0)만 있어서 `device=1`은 범위를 벗어납니다.

In [18]:
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

1
Tesla T4


In [19]:
import torch.nn as nn
layer=nn.Linear(2,2)
layer.cuda(0)

Linear(in_features=2, out_features=2, bias=True)

In [20]:
x=torch.cuda.FloatTensor(2,2)
x.device


device(type='cuda', index=0)